## Data Ingestion (Streaming)

**Importing Useful Libraries**

In [1]:
import os
import requests
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

False

**Getting The Current Weather in Barcelona (Semi-structured)**

For starters, we will employ `Open-Mateo`, an open-source Weather API of free access: `https://open-meteo.com/`. The data we have here come from different parts of the world, if we want to get the current Weather for Barcelona, we need to provide the latitude and longitude of Barcelona to the URL. The data obtained is updated every few minutes and the default timezone is GMT.

Do note that the API maps our requested coordinates to the closest model grid point, so the coordinates it returns are not what we have given to it.

In [2]:
# We are interested in the current weather for Barcelona
latitude = 41.385
longitude = 2.173

# API endpoint
url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
response = requests.get(url) # Get the response from the API

# Check if request was successful
if response.status_code == 200:
    data = response.json()
    print("Raw Data:", data)
else:
    print("Failed to fetch data:", response.status_code)

Raw Data: {'latitude': 41.375, 'longitude': 2.125, 'generationtime_ms': 0.055670738220214844, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 29.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-03-07T13:45', 'interval': 900, 'temperature': 13.7, 'windspeed': 9.4, 'winddirection': 86, 'is_day': 1, 'weathercode': 61}}


**Getting The Current Air Quality in Barcelona (Semi-structured)**

In this case, we will use `Open_AQ`, an open-source Air Quality API for global air quality data: `https://api.openaq.org/`. It is also of free access, but for using it, we need to create a free account to obatin an `API_KEY` which is needed when requesting for the data. The data are updated less frequently than the previous API, in this case, its frequency can vary from minutes to hours. Moreover, for this specific API, the data are not coming from a single point of source, there are three different institutions providing the data continuously.

In [8]:
# We need to sign up an OpenAQ account
OPENAQ_API_KEY = os.environ.get("OPENAQ_API_KEY")
spain_country_id = 67 # Internal ID for Spain
search_city = "Barcelona"

# API endpoint
url = f"https://api.openaq.org/v3/locations?countries_id={spain_country_id}&limit=100"
response = requests.get(url, headers={"X-API-Key": OPENAQ_API_KEY})

if response.status_code == 200:
    data = response.json()
    results = data["results"]

    # Filter locally for city / locality containing "Barcelona"
    barcelona_locs = [
        loc for loc in results
        if isinstance(loc["locality"], str) and search_city.lower() in loc["locality"].lower().strip()
    ]

    print("Raw Data:", barcelona_locs)
else:
    print("Failed to fetch data:", response.status_code)

Raw Data: [{'id': 2990, 'name': "BARCELONA (PARC DE LA VALL D'HEBRON)", 'locality': 'BARCELONA', 'timezone': 'Europe/Madrid', 'country': {'id': 67, 'code': 'ES', 'name': 'Spain'}, 'owner': {'id': 4, 'name': 'Unknown Governmental Organization'}, 'provider': {'id': 70, 'name': 'EEA'}, 'isMobile': False, 'isMonitor': True, 'instruments': [{'id': 2, 'name': 'Government Monitor'}], 'sensors': [{'id': 7566, 'name': 'co µg/m³', 'parameter': {'id': 4, 'name': 'co', 'units': 'µg/m³', 'displayName': 'CO mass'}}, {'id': 4274999, 'name': 'no µg/m³', 'parameter': {'id': 19843, 'name': 'no', 'units': 'µg/m³', 'displayName': 'NO mass'}}, {'id': 6341, 'name': 'no2 µg/m³', 'parameter': {'id': 5, 'name': 'no2', 'units': 'µg/m³', 'displayName': 'NO₂ mass'}}, {'id': 4287829, 'name': 'nox µg/m³', 'parameter': {'id': 27, 'name': 'nox', 'units': 'µg/m³', 'displayName': 'NOx mass'}}, {'id': 6379, 'name': 'o3 µg/m³', 'parameter': {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': 'O₃ mass'}}, {'id': 7120